# 02 · Depth-5 GBDT streams on the public "Version 10" feature recipe

**Role in the stack:** the first partner that was *strong and different*. `lgb_bag2` (2 seeds) CV 0.94530, weight 0.060;
`xgb_bag2` 0.029; `cb_bag2` 0.014.

## Why a second GBDT at all
The additive model cannot use interactions by construction. A public notebook (lamhuy8904, LB 0.94608) used an unconstrained
depth-5 GBDT on ~60 engineered features with target encodings, and reached about the same score by a different route. On our
folds the two models correlate 0.99 but their errors differ enough that a stack of the two gave **+0.00035** over the additive
model alone. No NN, no third GBDT, no feature set ever added that much again.

## What `feats_v10.py` builds
- Label encodings of the 6 categoricals, raw numerics.
- Income structure flags: at the 30,000 floor, round-100, round-1000, trailing zeros, last digit, mod 100 / mod 1000, and
  integer divisions by 100 / 250 / 500 / 1000 / 2000 / 5000. The same for commute at 0.1 km resolution.
- Four explicit products (Env x Subsidy, Env x Income, Subsidy x Income, Anxiety x Commute) and charging totals.
- **Target encodings** of exact income, income//100, income//1000, commute, and five joint keys, with smoothing m=100 / 50.

## The honesty fix
The public notebook early-stops on the fold it scores, which inflates its OOF by ~0.0003. Our port
(`run_stream.py --model lgb|xgb|cb`) carves an inner 10% of the training fold for early stopping, so the scored fold never touches
the stopping decision. Target encodings are **nested**: fitted on the training part of each outer fold only, with inner 5-fold
cross-fitting for the training rows themselves (`add_te`). The OOF reads ~0.0003 lower than the notebook claims and lands on the
same leaderboard score.

In [ ]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
from cv import load, run_cv, paired, SEED, NFOLDS, OOF, TEST
X, y, Xte, te_id = load()          # data/train.csv, data/test.csv  (kaggle competitions download -c playground-series-s6e9)
print(X.shape, Xte.shape, 'positives %.4f' % y.mean())

In [ ]:
from feats_v10 import build, add_te
Dtr, Dte = build(X, Xte)      # static features for train and test (joint string keys kept only for TE)
print(Dtr.shape, 'columns:', list(Dtr.columns)[:12], '...')

In [ ]:
# Parameters, as used by run_stream.py. LightGBM is the public recipe's; XGB/CatBoost are our depth-5 ports.
P_LGB = dict(objective='binary', metric='auc', learning_rate=0.006, num_leaves=32, max_depth=5,
             min_child_samples=15, feature_fraction=0.30, bagging_fraction=0.85, bagging_freq=1,
             lambda_l1=0.071, lambda_l2=2.0, max_bin=2048, n_jobs=-1, verbose=-1)
P_XGB = dict(objective='binary:logistic', eval_metric='auc', tree_method='hist', device='cuda',
             learning_rate=0.008, max_depth=5, min_child_weight=20, subsample=0.85, colsample_bytree=0.70,
             colsample_bylevel=0.75, gamma=0.30, reg_alpha=0.10, reg_lambda=2.0, n_estimators=25000, early_stopping_rounds=400)
P_CB = dict(loss_function='Logloss', eval_metric='AUC', learning_rate=0.025, depth=5, l2_leaf_reg=3.0,
            iterations=12000, task_type='GPU', verbose=0)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

def fit_lgb(Xtr, ytr, Xva, Xt, seed):
    # inner 10% split for early stopping -> the outer validation fold is never seen during training
    Xa, Xb, ya, yb = train_test_split(Xtr, ytr, test_size=0.1, stratify=ytr, random_state=seed)
    m = lgb.train({**P_LGB, 'random_state': seed}, lgb.Dataset(Xa, ya), 30000,
                  valid_sets=[lgb.Dataset(Xb, yb)], callbacks=[lgb.early_stopping(400, verbose=False)])
    return m.predict(Xva, num_iteration=m.best_iteration), m.predict(Xt, num_iteration=m.best_iteration), m.best_iteration

seed = 42; yv = y.values
oof, tpred, iters = np.zeros(len(y)), np.zeros(len(Xte)), []
for f, (itr, iva) in enumerate(StratifiedKFold(NFOLDS, shuffle=True, random_state=SEED).split(X, y)):
    Xtr, (Xva, Xt) = add_te(Dtr.iloc[itr], yv[itr], [Dtr.iloc[iva], Dte])   # nested target encoding
    oof[iva], tp, it = fit_lgb(Xtr, yv[itr], Xva, Xt, seed); tpred += tp / NFOLDS; iters.append(it)
    print('fold%d auc=%.5f iters=%d' % (f, roc_auc_score(y.iloc[iva], oof[iva]), it))
print('oof=%.5f' % roc_auc_score(y, oof))
np.save(OOF / 'oof_lgb_s42.npy', oof); np.save(TEST / 'test_lgb_s42.npy', tpred)

## Seeds and refits
`run_stream.py --model lgb --seed 202` gives the second seed. `--refit` retrains on 100% of train with the mean early-stopped round
count x 1.1 and writes `test_<tag>_full.npy`, which the stacker prefers over the fold-average. In the stack, all seeds of one family are
averaged in logit space into one column (`lgb_bag2`) **before** fitting weights, because near-duplicate seed columns let the
stacker fit noise with +/- weights on the same model.

## What we learned
- Same-library seeds are near-duplicates: extra seeds reduce variance, they do not add diversity.
- XGBoost and CatBoost on the same features earned ~0 weight once LightGBM was in: three libraries, one signal.
- Leave-one-out on the 09-05 stack: without the additive model -0.00038; without this one -0.00002, because TabM/XGB/CB
  substitute for it. The stack at that point was really two models: the additive tree plus one unconstrained GBDT.